# Assignment 1: AI-Powered Intrusion Detection System
## Course: Information Security | CLO 4
**Author:** Abdul Rehman  
**Dataset:** CIC-IDS2017 (Real Network Traffic Dataset – 2.5M flows)  
**Models:** Random Forest & Decision Tree  
**Goal:** Classify network traffic as Normal or one of 6 attack types

## 1. Setup & Imports

In [ ]:
import warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score, f1_score)
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid", font_scale=1.1)
print("All libraries loaded successfully ✓")

## 2. Dataset Loading & EDA

### Dataset: CIC-IDS2017 (Canadian Institute for Cybersecurity)
The **CIC-IDS2017** dataset was chosen because:
- Contains **realistic network traffic** captured over 5 days (Mon–Fri)
- Includes **6 real attack categories**: DoS, DDoS, Port Scanning, Brute Force, Web Attacks, Bots
- Widely used as a **benchmark** in NIDS research (1,000+ citations)
- Represents threats directly relevant to enterprise environments like SecureNet Corp.

In [ ]:
# Load the CIC-IDS2017 cleaned dataset
DATA_PATH = "./dataset/cicids2017_cleaned.csv"

print("Loading full dataset...")
t0 = time.time()
df_full = pd.read_csv(DATA_PATH)
df_full.columns = df_full.columns.str.strip()
df_full = df_full.rename(columns={"Attack Type": "Label"})

print(f"Full Dataset Shape: {df_full.shape}")
print(f"Load time: {time.time()-t0:.1f}s")
print("\nLabel Distribution (Full Dataset):")
print(df_full["Label"].value_counts())
print(f"\nMissing Values: {df_full.isnull().sum().sum()}")

In [ ]:
# Stratified sample of 50,000 rows for efficient training
_, df = train_test_split(df_full, test_size=50000, stratify=df_full["Label"], random_state=42)
df = df.reset_index(drop=True)
print(f"Working sample: {df.shape}")
print("\nSample Label Distribution:")
print(df["Label"].value_counts())

In [ ]:
# EDA – Class Distribution (Full Dataset)
fig, ax = plt.subplots(figsize=(10, 5))
vc = df_full["Label"].value_counts()
colors = ["#4CAF50","#F44336","#2196F3","#FF9800","#9C27B0","#00BCD4","#FF5722"]
bars = ax.bar(vc.index, vc.values, color=colors, edgecolor="white", linewidth=1.3)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.05,
            f'{int(b.get_height()):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_yscale("log")
ax.set_title("CIC-IDS2017: Network Traffic Class Distribution (2,520,751 flows)", fontsize=14, fontweight="bold")
ax.set_xlabel("Traffic Class"); ax.set_ylabel("Count (log scale)")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

In [ ]:
# EDA – Correlation Heatmap (Top Features by Variance)
top_feats = df.select_dtypes(include=np.number).var().nlargest(12).index.tolist()
fig, ax = plt.subplots(figsize=(11, 9))
corr = df[top_feats].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, annot=True,
            fmt=".2f", linewidths=0.4, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Heatmap – Top-12 High-Variance Features", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Data Preprocessing

Steps performed:
1. **Infinity/NaN Removal** – Replace ±inf with NaN, drop remaining NaN rows  
2. **Duplicate Removal** – Drop exact duplicate flow records  
3. **Label Encoding** – Convert string labels to integer classes  
4. **Feature Scaling** – StandardScaler (zero mean, unit variance) for numerical stability  
5. **Train/Test Split** – 80/20 stratified split  
6. **SMOTE** – Synthetic Minority Over-sampling to address class imbalance (Bots: 39 → balanced)

In [ ]:
# Preprocessing
before = len(df)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
print(f"Rows removed: {before - len(df)} | Remaining: {len(df):,}")

le = LabelEncoder()
df["Label_enc"] = le.fit_transform(df["Label"])
print(f"Classes: {list(le.classes_)}")

feature_cols = [c for c in df.columns if c not in ["Label","Label_enc"]]
X = df[feature_cols].values
y = df["Label_enc"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

sm = SMOTE(random_state=42, k_neighbors=3)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(f"After SMOTE: {X_train_res.shape[0]:,} samples")

## 4. Model Development

### Model 1: Random Forest Classifier
**Justification:** Random Forest was selected as the primary model because:
- Handles **high-dimensional network data** (52 features) excellently
- Naturally **resistant to overfitting** via ensemble averaging
- Provides **feature importance** scores for interpretability — critical for CISO reporting
- Consistently achieves **state-of-the-art** performance on IDS benchmarks

### Model 2: Decision Tree Classifier (Baseline)
**Justification:** Used as an interpretable baseline for comparison. A single tree is faster and
fully explainable, though typically less accurate than an ensemble.

In [ ]:
# Train Random Forest
print("Training Random Forest...")
t1 = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42,
                            n_jobs=-1, class_weight="balanced")
rf.fit(X_train_res, y_train_res)
rf_preds = rf.predict(X_test)
print(f"✓ Random Forest trained in {time.time()-t1:.1f}s")

# Train Decision Tree
print("\nTraining Decision Tree...")
t2 = time.time()
dt = DecisionTreeClassifier(max_depth=15, random_state=42, class_weight="balanced")
dt.fit(X_train_res, y_train_res)
dt_preds = dt.predict(X_test)
print(f"✓ Decision Tree trained in {time.time()-t2:.1f}s")

## 5. Evaluation & Security Analysis

In [ ]:
# Metrics helper
def evaluate(name, preds):
    acc  = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average="weighted", zero_division=0)
    rec  = recall_score(y_test, preds, average="weighted", zero_division=0)
    f1   = f1_score(y_test, preds, average="weighted", zero_division=0)
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"\n  Per-Class Report:")
    print(classification_report(y_test, preds, target_names=le.classes_, zero_division=0))
    return acc, prec, rec, f1

rf_acc, rf_prec, rf_rec, rf_f1 = evaluate("Random Forest Classifier", rf_preds)
dt_acc, dt_prec, dt_rec, dt_f1 = evaluate("Decision Tree Classifier", dt_preds)

In [ ]:
# Confusion Matrix – Random Forest
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, preds, title, cmap in zip(
        axes,
        [rf_preds, dt_preds],
        ["Random Forest", "Decision Tree"],
        ["Blues", "Oranges"]):
    cm = confusion_matrix(y_test, preds)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
    annot = np.array([[f'{v:.1f}%' for v in row] for row in cm_pct])
    sns.heatmap(cm_pct, annot=annot, fmt='', cmap=cmap,
                xticklabels=le.classes_, yticklabels=le.classes_,
                ax=ax, linewidths=0.5, linecolor='white')
    ax.set_title(f"Confusion Matrix – {title}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.suptitle("Confusion Matrices (% of True Class)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Feature Importance
fig, ax = plt.subplots(figsize=(10, 6))
fi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)[:15]
colors_fi = plt.cm.viridis(np.linspace(0.2, 0.9, len(fi)))
ax.barh(fi.index[::-1], fi.values[::-1], color=colors_fi[::-1])
ax.set_title("Top-15 Feature Importances – Random Forest", fontsize=13, fontweight="bold")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout(); plt.show()

In [ ]:
# Model Comparison Chart
fig, ax = plt.subplots(figsize=(8, 5))
met_keys = ["Accuracy","Precision","Recall","F1-Score"]
rf_vals = [rf_acc, rf_prec, rf_rec, rf_f1]
dt_vals = [dt_acc, dt_prec, dt_rec, dt_f1]
x = np.arange(len(met_keys)); w = 0.32
b1 = ax.bar(x-w/2, rf_vals, w, label="Random Forest", color="#1565C0", edgecolor="white")
b2 = ax.bar(x+w/2, dt_vals, w, label="Decision Tree",  color="#E65100", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(met_keys, fontsize=12)
ax.set_ylim(max(0, min(rf_vals+dt_vals)-0.05), 1.02)
ax.set_title("Model Performance Comparison – CIC-IDS2017 Real Dataset", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.set_ylabel("Score"); ax.grid(axis='y', alpha=0.4)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Per-Class Recall – Critical Security Metric
from sklearn.metrics import classification_report as cr
rf_report = cr(y_test, rf_preds, target_names=le.classes_, zero_division=0, output_dict=True)
dt_report = cr(y_test, dt_preds, target_names=le.classes_, zero_division=0, output_dict=True)

classes = le.classes_
rf_recalls = [rf_report[c]["recall"] for c in classes]
dt_recalls = [dt_report[c]["recall"] for c in classes]

fig, ax = plt.subplots(figsize=(10, 5))
x3 = np.arange(len(classes)); w3 = 0.35
ax.bar(x3-w3/2, rf_recalls, w3, label="Random Forest", color="#1565C0")
ax.bar(x3+w3/2, dt_recalls, w3, label="Decision Tree",  color="#E65100")
ax.axhline(0.9, color='red', linestyle='--', linewidth=1.5, label="90% Recall Threshold")
ax.set_xticks(x3); ax.set_xticklabels(classes, rotation=20, ha='right', fontsize=11)
ax.set_ylim(0, 1.15); ax.set_ylabel("Recall"); ax.legend()
ax.set_title("Per-Class Recall – Security-Critical Metric\n(High recall = fewer missed real attacks)",
             fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

print("\nSECURITY ANALYSIS:")
print("─" * 50)
for c, rr, dr in zip(classes, rf_recalls, dt_recalls):
    flag = "⚠️ LOW" if rr < 0.9 else "✓ OK"
    print(f"  {c:15s}  RF Recall: {rr:.3f}  DT Recall: {dr:.3f}  {flag}")

## 6. Conclusion

### Key Findings
| Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| **Random Forest** | **99.43%** | **99.71%** | **99.43%** | **99.56%** |
| Decision Tree | 99.03% | 99.41% | 99.03% | 99.20% |

### Security Implications
- **DDoS/DoS attacks** are detected with near-perfect recall (~100%) — critical for availability protection
- **Port Scanning** achieves 100% recall — unauthorized network reconnaissance is fully caught
- **Bots** show lower recall (62%) due to extreme class imbalance (only 1,948 of 2.5M flows) — a known challenge requiring further feature engineering or deep learning
- **Random Forest outperforms Decision Tree** on all metrics, validating the ensemble approach

### Recommendation to CISO
Deploy the Random Forest model as a real-time traffic classifier integrated with the existing NIDS. 
Priority for improvement: collect more Bot traffic samples and explore LSTM-based temporal modeling 
to capture behavioral patterns missed by flow-level features alone.